# PBAIH12-B 實作二：譜庫比對與假命中

**目標**：實作一我們看過怎麼比較兩張 MS/MS 譜。這份實作做真正的**譜庫比對**：把一個實驗室測的譜當成「未知」，去**其他實驗室**的譜庫裡找最像的，回答三個問題：
1. 化合物在庫裡時，最高分的候選是不是對的？
2. 化合物**不在**庫裡時，會不會還是有人拿到高分？
3. 設門檻、換加權方式，會改變什麼？

**資料**：和實作一相同的 MassBank 2026.11 學生子集（CC BY／CC0、正離子 [M+H]+、標準品確認，約 4.9 萬筆、24 個貢獻者）。來源：Zenodo doi:10.5281/zenodo.24082491；轉載請標示 MassBank 與作者。

| 部分 | 內容 |
|---|---|
| Part 0 | 環境與資料（**上課一開始先執行**） |
| Part 1 | 建庫與查詢集：leave-one-contributor-out（練習 1） |
| Part 2 | 對一筆查詢找最像的候選（練習 2） |
| Part 3 | 批次評估：top-1、top-3、庫外命中（練習 3） |
| Part 4 | 門檻與匹配峰數的取捨（練習 4） |
| Part 5 | 加權比較（練習 5） |
| Part 6 | 看失敗案例（練習 6） |
| 延伸 E1–E3 | modified cosine、熵相似度、reverse dot product 與 presence ratio（**課後／有餘力**） |

> 這裡的 **Part** 是 notebook 的章節；「第 2 節」「第 3 節」指上課的第幾節。

每個 Part 都分成：**為什麼** → **怎麼做** → **看結果** → **想一想**。練習題是「補完這一行」，空格寫成 `____`。

🆘 **卡住時**：① 先讀錯誤訊息的最後一行（通常會說是哪個變數或哪一行）→ ② 問 Colab 內建的 Gemini（把錯誤訊息貼給它，請它「用中文解釋」）→ ③ 打開下方的 💡 提示 → ④ 舉手問老師或助教。

## 執行環境

本 notebook 以 Google Colab 為準（Python 3.13），需要 `matchms==1.0.0`（Part 0-a 會安裝）。圖表文字用英文，因為 Colab 預設沒有中文字型。

---
# 核心路徑（Part 0–2 在上課第 2 節後段，Part 3–6 在第 3 節）

## Part 0. 環境與資料（上課一開始先執行）

**怎麼做**：依序執行 0-a（安裝）和 0-b（載入套件與資料）。**每一份 notebook 都要各自執行一次 Part 0**（Colab 的每份 notebook 是獨立的執行環境，不會共用實作一已安裝的套件和下載的資料）。若 0-a 要你重新啟動工作階段，照做後**直接執行 0-b**。

In [ ]:
# 0-a 安裝 matchms（只在 Colab 上需要；上課一開始先執行這一格）
import sys
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q matchms==1.0.0")

# 安裝可能會升級 numpy／pandas／matplotlib；如果這些套件在安裝前已經被載入，記憶體裡還是舊版，要重新啟動才會換新
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    print("⚠️ 這些套件已升級，但記憶體裡還是舊版：", stale)
    print("請按上方選單「執行階段 → 重新啟動工作階段」，然後【直接執行 0-b】（不用再執行 0-a）。")
else:
    print("安裝完成，請執行 0-b。")

In [ ]:
# 0-b 載入套件、取得資料、讀入譜庫子集
import os, sys, glob, zipfile, logging
import importlib.metadata as md

IN_COLAB = "google.colab" in sys.modules
stale = [p for p in ("numpy", "pandas", "matplotlib") if p in sys.modules and md.version(p) != sys.modules[p].__version__]
if stale:
    raise RuntimeError(f"{stale} 的版本在安裝後改變了。請按「執行階段 → 重新啟動工作階段」，再重新執行這一格。")

# 取得資料：優先順序 = 環境變數 PBAIH12_DATA_DIR → 已下載（或手動放好）的資料夾 → 從課程 GitHub 下載
DATA_URL = "https://github.com/wang3yuan/PBAIH/releases/download/data-v2/PBAIH12_data.zip"
DATA_ROOT = "pbaih12_data"

def find_data_dir(*roots):
    """在這些資料夾底下找含有 massbank_pos_ms2.parquet 的資料夾。"""
    for root in roots:
        hits = sorted(glob.glob(os.path.join(root, "**", "massbank_pos_ms2.parquet"), recursive=True))
        if hits:
            return os.path.dirname(hits[0])
    return None

DATA_DIR = os.environ.get("PBAIH12_DATA_DIR") or find_data_dir(DATA_ROOT, ".")
if DATA_DIR is None:
    import urllib.request
    os.makedirs(DATA_ROOT, exist_ok=True)
    zip_path = os.path.join(DATA_ROOT, "PBAIH12_data.zip")
    print("下載中（約 10 MB）...")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)
    DATA_DIR = find_data_dir(DATA_ROOT)
assert DATA_DIR is not None, "找不到資料，請看 Part 0.1"

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", message=".*hdf5plugin.*")   # matchms 內部套件的無害提示
import matchms
from matchms import Spectrum
from matchms.similarity import CosineGreedy
logging.getLogger("matchms").setLevel(logging.ERROR)   # 關掉 matchms 的警告訊息

for name, mod in [("matchms", matchms), ("numpy", np), ("pandas", pd), ("matplotlib", matplotlib)]:
    print(f"{name:11s} {mod.__version__}")
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| 本機")

lib = pd.read_parquet(os.path.join(DATA_DIR, "massbank_pos_ms2.parquet"))
meta_ms2 = pd.read_parquet(os.path.join(DATA_DIR, "massbank_ms2_meta.parquet"))
print(f"譜庫子集：{len(lib):,} 筆 MS/MS、{lib.ik14.nunique():,} 個化合物、{lib.contributor.nunique()} 個貢獻者")
print("授權：", lib.license.value_counts().to_dict())


# 前處理：把一筆紀錄變成 matchms 的 Spectrum。兩個步驟都寫在這裡，不交給套件預設去做：
#   (1) 先移除前驅物附近的峰（m/z ≥ 前驅物 m/z − 1.6）：沒有碎裂的前驅物離子沒有結構資訊，而且常常是最大的峰
#   (2) 再去掉低於（剩下的）基峰 1% 的峰，當作雜訊
PREC_OFFSET = -1.6
def get_spec(r, min_rel=0.01):
    mz = np.asarray(r.mz, dtype=float); it = np.asarray(r.intensity, dtype=float)
    keep = mz < r.precursor_mz + PREC_OFFSET
    mz, it = mz[keep], it[keep]
    if len(mz):
        keep = it >= min_rel * it.max()
        mz, it = mz[keep], it[keep]
    return Spectrum(mz=mz, intensities=it, metadata={"precursor_mz": float(r.precursor_mz)})

### Part 0.1 如果安裝或下載失敗怎麼辦

| 狀況 | 怎麼辦 |
|---|---|
| 0-a 印出「這些套件已升級，但記憶體裡還是舊版」 | 按「執行階段 → 重新啟動工作階段」，然後**直接執行 0-b**（不用再執行 0-a）。 |
| 0-a 跑完出現紅字的 `ERROR: pip's dependency resolver ...`，提到 `pandas`、`requests` 或 `google-colab` | 通常可以忽略；只要 0-b 能順利跑完就好。 |
| `pip` 顯示 `No matching distribution found for matchms==1.0.0` | 套件版本和 Python 不合。執行 `!python --version`，把結果告訴老師或助教。 |
| 0-b 顯示 `RuntimeError ... 請按「執行階段 → 重新啟動工作階段」` | 照訊息做：重新啟動後再執行 0-b。 |
| 下載時顯示 `HTTP Error`（例如 404、429）或連線失敗 | 用瀏覽器打開 https://github.com/wang3yuan/PBAIH/releases/tag/data-v2 ，手動下載 `PBAIH12_data.zip`，在 Colab 左側「檔案」面板上傳，再執行下面「手動放資料」那段。 |
| 網路逾時、下載到一半中斷 | 刪掉 `pbaih12_data` 資料夾後再執行一次 0-b。 |

手動放資料（上傳 zip 之後）：

```python
import zipfile
zipfile.ZipFile("PBAIH12_data.zip").extractall("pbaih12_data")
```
然後重新執行 0-b。

> **每一份 notebook 都要各自執行一次 Part 0**（Colab 的每份 notebook 是獨立的執行環境，不會共用已安裝的套件和下載的資料）。

## Part 1. 建庫與查詢集：leave-one-contributor-out

**為什麼**：如果拿一張譜去查「包含它自己」的譜庫，一定查得到，分數是 1.000，這沒有意義。真實情況是：你的樣品是在**你的**實驗室、用**你的**儀器和碰撞能量測的，譜庫裡的譜來自**別人**。

所以我們這樣設計：選一個貢獻者（實驗室）當**查詢**，它的譜假裝是「未知」；**譜庫只放其他貢獻者的譜**。因為我們知道每筆譜的真正化合物（`ik14`），就能判斷命中對不對。

另外，查詢的化合物可能根本**不在**其他實驗室的庫裡。這種情況在現實中很常見，我們也把它分出來看。

**前處理**：和實作一一樣，用 Part 0-b 定義的 `get_spec`：先移除**前驅物附近的峰**（m/z ≥ 前驅物 − 1.6，沒碎裂的前驅物沒有結構資訊），再去掉低於基峰 1% 的峰。**如果一張譜只剩前驅物那個峰，前處理後就沒有峰了**，這種查詢永遠找不到候選（低碰撞能量的譜常常這樣）。

In [ ]:
# 1-1 各貢獻者概況（前 10 名）
summary = (lib.groupby("contributor")
           .agg(n_spectra=("accession", "size"), n_compounds=("ik14", "nunique"),
                instrument=("instrument_type", lambda s: s.value_counts().index[0]))
           .sort_values("n_spectra", ascending=False))
display(summary.head(10))

QUERY_LAB = "Athens_Univ"      # 當成「未知」的實驗室；可改成 "BAFG"、"LCSB" 再跑一次（名稱要和上表完全一樣；改了之後從 Part 1 依序重跑）
N_QUERY = 1000                 # 抽幾筆當查詢（幾秒就跑完；改小會更快、但統計更不穩）
MS1_TOL = 0.01                 # 前驅物 m/z 篩候選的範圍（Da）；沿用 MS-DIAL 預設
MS2_TOL = 0.025                # 碎片 m/z tolerance（Da）；沿用 MS-DIAL 預設
assert QUERY_LAB in set(lib.contributor), "QUERY_LAB 的名稱要和上表的貢獻者完全一樣（大小寫也要）"

### 練習 1：切出查詢集和譜庫

In [ ]:
query_all = lib[lib.contributor == ____]
ref = lib[lib.contributor != QUERY_LAB].sort_values("precursor_mz").reset_index(drop=True)
N_QUERY = min(N_QUERY, len(query_all))                 # 有些實驗室的譜不到 N_QUERY 筆
query = query_all.sample(N_QUERY, random_state=0).reset_index(drop=True)

ref_iks = set(ref.ik14)
query["in_ref"] = query.ik14.isin(ref_iks)      # 這個化合物在別人的庫裡嗎？
print(f"查詢：{QUERY_LAB} 抽 {len(query)} 筆；譜庫：其他 {ref.contributor.nunique()} 個貢獻者的 {len(ref):,} 筆")
print(f"查詢中，化合物在譜庫裡的：{query.in_ref.sum()} 筆；不在的：{(~query.in_ref).sum()} 筆")

<details><summary>💡 練習 1 提示（先自己試，卡住再打開）</summary>

查詢集是「貢獻者等於 `QUERY_LAB`」的列；譜庫是「不等於」的列（已經寫好了）。

<details><summary>參考答案</summary>

```python
query_all = lib[lib.contributor == QUERY_LAB]
```
</details>
</details>

In [ ]:
# 1-2 把每一列轉成 matchms 的 Spectrum（前處理見上；get_spec 在 Part 0-b 定義）
ref_specs = [get_spec(r) for _, r in ref.iterrows()]
q_specs = [get_spec(r) for _, r in query.iterrows()]
ref_mz = ref.precursor_mz.values                       # 已排序，之後用 searchsorted 快速找範圍
n_empty = sum(len(s.peaks.mz) == 0 for s in q_specs)
print("完成：", len(ref_specs), "張譜庫譜，", len(q_specs), "張查詢譜")
print(f"其中 {n_empty} 張查詢譜前處理後沒有任何峰（只剩前驅物），它們找不到候選")

**想一想**
1. 為什麼譜庫要排除「查詢實驗室」的所有譜，而不是只排除那一筆？
2. 查詢裡有些化合物**不在**別人的庫裡。如果這些查詢拿到高分，那是對的嗎？

## Part 2. 對一筆查詢找最像的候選

**為什麼**：譜庫比對的流程是：① 用前驅物 m/z 篩出候選（只比質量差在 `MS1_TOL` 內的譜，其他不用算）→ ② 對每個候選算相似度 → ③ 由高到低排序。

**怎麼做**：下面的函式 `search_one` 做這三步。每個候選回傳**分數**和**對上的峰數**（matches）；**至少要對上 1 個碎片峰**才算候選（完全沒有碎片對上、分數 0 的不算）。

In [ ]:
def search_one(qi, sim, topn=10):
    """查詢 q_specs[qi]：前驅物 m/z ± MS1_TOL 內的所有譜庫譜都算 cosine，回傳依分數排序的前 topn 名。
    至少要對上 1 個碎片峰才算候選（分數 0、沒有任何碎片對上的不算，不然同分時會被當成「命中」）。"""
    q = q_specs[qi]
    pmz = query.precursor_mz[qi]
    lo, hi = np.searchsorted(ref_mz, [pmz - MS1_TOL, pmz + MS1_TOL])    # 候選的範圍
    hits = []
    for j in range(lo, hi):
        res = sim.pair(____, q)
        if int(res["matches"]) < 1:
            continue
        hits.append((float(res["score"]), int(res["matches"]), ref.ik14[j], j))
    hits.sort(reverse=True)
    return hits[:topn]

# 常用設定：強度開根號、m/z 一次方加權；remove_precursor=False 因為前驅物附近的峰已在 get_spec 移除
sim = CosineGreedy(tolerance=MS2_TOL, mz_power=1, intensity_power=0.5, remove_precursor=False)

<details><summary>💡 練習 2 提示（先自己試，卡住再打開）</summary>

`sim.pair(a, b)` 是拿兩張譜 a、b 算相似度。這裡 a 是譜庫裡第 `j` 張譜 `ref_specs[j]`，b 是查詢譜 `q`。

<details><summary>參考答案</summary>

```python
res = sim.pair(ref_specs[j], q)
```
</details>
</details>

In [ ]:
# 2-2 找一筆「化合物在庫裡、第 1 名正確、分數 ≥ 0.8、而且前 10 名裡有不同化合物」的查詢來看
for qi in np.where(query.in_ref)[0]:
    qi = int(qi)
    top = search_one(qi, sim)
    if top and top[0][2] == query.ik14[qi] and top[0][0] >= 0.8 and len({t[2] for t in top}) >= 2:
        break
lo, hi = np.searchsorted(ref_mz, [query.precursor_mz[qi] - MS1_TOL, query.precursor_mz[qi] + MS1_TOL])
print(f"查詢：{query['name'][qi]}（{query.formula[qi]}），前驅物 m/z {query.precursor_mz[qi]:.4f}，CE {query.collision_energy[qi]}")
print(f"前驅物 ±{MS1_TOL} Da 內有 {hi - lo} 張譜庫譜；下面是分數最高的前 {len(top)} 名")
show = pd.DataFrame([{"rank": k + 1, "name": ref["name"][j], "formula": ref.formula[j], "score": s, "matches": m,
                      "correct": ik == query.ik14[qi]} for k, (s, m, ik, j) in enumerate(top)])
display(show.round(3))

**看結果**：第 1 名是不是正確的化合物？分數多高？對上幾個峰？注意：同一個化合物在不同實驗室可能有不同的名稱（所以我們用 `ik14` 判斷對錯，不用名稱），而且同一個化合物在庫裡常有好幾筆譜（不同碰撞能量），所以前幾名可能都是同一個化合物。**不同**化合物的候選分數離第 1 名有多遠？如果很接近，你有多大把握？

## Part 3. 批次評估：top-1、top-3、庫外命中

**為什麼**：看一兩筆查詢不夠，要對整批查詢統計：

- **top-1 正確率**＝「第 1 名就是正確化合物」的查詢數 ÷ 化合物在庫裡的查詢數。
- **top-3 正確率**：正確化合物出現在**前 3 個不同化合物**裡的比例（同一化合物的多張譜只算一次）。
- **庫外命中**：化合物**不在**庫裡的查詢，仍然有候選的比例，以及最高分有多高。

**怎麼做**：`run_search` 對全部查詢跑一次 `search_one`（用指定的 m/z 與強度加權），回傳每筆查詢的結果清單。

In [ ]:
def run_search(setting=(1, 0.5), topn=30):
    """setting = (mz_power, intensity_power)。回傳每筆查詢的前 topn 名候選。"""
    mp, ip = setting
    s = CosineGreedy(tolerance=MS2_TOL, mz_power=mp, intensity_power=ip, remove_precursor=False)
    out = []
    for qi in range(len(query)):
        out.append({"qi": qi, "true": query.ik14[qi], "in_ref": bool(query.in_ref[qi]), "top": search_one(qi, s, topn)})
    return out

import time
t0 = time.time()
res = run_search((1, 0.5))
print(f"完成 {len(res)} 筆查詢，用時 {time.time() - t0:.1f} 秒")

### 練習 3：算 top-1 與 top-3 正確率

每筆結果的 `r["top"]` 是依分數排序的候選清單，每個候選是 `(分數, 對上峰數, 化合物 ik14, 譜庫編號)`；所以 `r["top"][0][2]` 就是第 1 名的化合物。

In [ ]:
def top_compounds(r, k):
    """前 k 個不同的化合物（同一化合物的多張譜只算一次）。"""
    out = []
    for t in r["top"]:
        if t[2] not in out:
            out.append(t[2])
        if len(out) == k:
            break
    return out

def evaluate(res):
    inr = [r for r in res if r["in_ref"]]
    out = [r for r in res if not r["in_ref"]]
    top1 = np.mean([bool(r["top"]) and r["top"][0][2] == ____ for r in inr])
    top3 = np.mean([r["true"] in top_compounds(r, 3) for r in inr])
    nocand = np.mean([len(r["top"]) == 0 for r in inr])
    out_called = np.mean([len(r["top"]) > 0 for r in out]) if out else float("nan")
    out_high = np.mean([bool(r["top"]) and r["top"][0][0] >= 0.7 for r in out]) if out else float("nan")
    return {"n_in_lib": len(inr), "n_not_in_lib": len(out), "top1": top1, "top3": top3,
            "in lib: no candidate": nocand, "not in lib: has a hit": out_called,
            "not in lib: top score >= 0.7": out_high}

summary3 = pd.Series(evaluate(res))
display(summary3.round(3).to_frame("value"))

<details><summary>💡 練習 3 提示（先自己試，卡住再打開）</summary>

第 1 名的化合物是 `r["top"][0][2]`，正確答案存在 `r["true"]`，兩者相等就算 top-1 命中。

<details><summary>參考答案</summary>

```python
top1 = np.mean([bool(r["top"]) and r["top"][0][2] == r["true"] for r in inr])
```
</details>
</details>

**看結果**：
- 化合物在庫裡的查詢，top-1 大約八成多。為什麼不是 100%？有一成多的庫內查詢**根本沒有候選**（`in lib: no candidate`）：前驅物只剩前驅物那個峰、或沒有任何碎片對上，所以 top-3 也救不回來。
- **化合物不在庫裡**的查詢，有多少比例仍然拿到某個候選（`not in lib: has a hit`）？其中最高分 ≥ 0.7 的有多少？（兩個比例的分母都是**全部**庫外查詢。）
- 這代表什麼？一個分數 0.9 的命中，你怎麼知道它屬於「對的」還是「庫外的假命中」？

**想一想**：真實樣品裡，你**不會知道**化合物在不在庫裡。這個統計對你實際判讀有什麼啟示？

## Part 4. 門檻與匹配峰數的取捨

**為什麼**：只看分數最高的候選，庫外的查詢也會「命中」。常見做法是加兩道門檻：

1. **分數門檻**：分數低於 `cutoff` 就不下判斷。
2. **匹配峰數**：對上的峰數少於 `min_matches` 就不下判斷（只對上 1–2 個峰卻得高分，常是假命中）。

代價是：門檻越嚴，「**未判定**」越多。我們來量化這個取捨。（這裡的分數是 matchms 的 cosine，**沒有平方**；MS-DIAL 的 dot product 是 cosine 的平方，所以 cosine 0.7 相當於 MS-DIAL 的 0.49。）

### 練習 4：寫出判斷規則

In [ ]:
def call(r, min_matches, cutoff):
    """依規則找第一個合格的候選（top 已依分數由高到低排序）；沒有合格的就回傳 None（未判定）。"""
    for t in r["top"]:                     # t = (分數, 對上峰數, ik14, 譜庫編號)
        if t[1] >= min_matches and t[0] >= ____:
            return t
    return None

def threshold_table(res, mins=(1, 3, 5), cuts=(0.0, 0.7, 0.9)):
    rows = []
    for mm in mins:
        for cut in cuts:
            inr = [(r, call(r, mm, cut)) for r in res if r["in_ref"]]
            out = [(r, call(r, mm, cut)) for r in res if not r["in_ref"]]
            rows.append({"min_matches": mm, "cutoff": cut,
                         "in lib: correct": np.mean([t is not None and t[2] == r["true"] for r, t in inr]),
                         "in lib: wrong": np.mean([t is not None and t[2] != r["true"] for r, t in inr]),
                         "in lib: no call": np.mean([t is None for r, t in inr]),
                         "not in lib: false call": np.mean([t is not None for r, t in out]) if out else float("nan")})
    return pd.DataFrame(rows)

tt = threshold_table(res)
display(tt.round(3))

<details><summary>💡 練習 4 提示（先自己試，卡住再打開）</summary>

`t[0]` 是分數、`t[1]` 是對上的峰數。兩個條件都要成立，所以寫 `t[1] >= min_matches and t[0] >= cutoff`。

<details><summary>參考答案</summary>

```python
if t[1] >= min_matches and t[0] >= cutoff:
```
</details>
</details>

In [ ]:
# 4-2 畫出取捨：固定 min_matches = 3，掃描分數門檻
cuts = np.round(np.arange(0, 1.01, 0.05), 2)
rows = []
for cut in cuts:
    inr = [(r, call(r, 3, cut)) for r in res if r["in_ref"]]
    out = [(r, call(r, 3, cut)) for r in res if not r["in_ref"]]
    rows.append({"cut": cut,
                 "correct": np.mean([t is not None and t[2] == r["true"] for r, t in inr]),
                 "no call": np.mean([t is None for r, t in inr]),
                 "false call (not in lib)": np.mean([t is not None for r, t in out]) if out else np.nan})
curve = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(7, 4))
for c, col in zip(["correct", "no call", "false call (not in lib)"], ["#0F8B8D", "#8D99AE", "#EE6C4D"]):
    ax.plot(curve.cut, curve[c] * 100, "o-", ms=3, label=c, color=col)
ax.set_xlabel("score cutoff (min matched peaks = 3)"); ax.set_ylabel("% of queries")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 你來決定門檻

下面兩個數字由你選。目標：**庫外假命中盡量低（例如 ≤ 2%），同時庫內「正確」盡量高**。改完執行，看結果，寫下你的選擇和理由（沒有標準答案，這就是實際工作中的取捨）。

In [ ]:
MY_MIN_MATCHES = 3     # 改改看：1、3、5、8 ...
MY_CUTOFF = 0.7        # 改改看：0.5、0.7、0.8、0.9 ...
mine = threshold_table(res, mins=(MY_MIN_MATCHES,), cuts=(MY_CUTOFF,))
display(mine.round(3))

**想一想**
1. 如果你的研究最怕「把錯的名字寫進論文」，你會選哪一組門檻？代價是什麼？
2. 如果你只是要先篩出可能的候選、之後再用標準品確認，門檻可以怎麼放？
3. 「未判定」的 feature 就沒有價值嗎？它們在報告裡該怎麼寫？（回想講課的信心等級。）

## Part 5. 加權比較

**為什麼**：講課提到 Stein & Scott（1994）的經典結果：把每個峰的權重設為 **W ＝ 強度ⁿ × m/zᵐ**，讓高質量碎片和中小峰更有影響力，在 EI 質譜的 NIST 資料上，rank 1 正確率從 69.9%（m＝0、n＝0.5）提高到最佳設定的 74.9%（m＝3、n＝0.6）。LC-MS/MS 也是這樣嗎？

**怎麼做**：matchms 的 `CosineGreedy` 有兩個參數 `mz_power`（m）和 `intensity_power`（n）。我們用四種設定各跑一次，比較 top-1。

**注意**：樣本數有限，top-1 的**抽樣誤差**不能忽略。下面用二項分布的標準誤（√(p(1−p)/n)）當粗略參考；同一個化合物有好幾張譜，所以真實的不確定性比這個數字**更大**。

### 練習 5：把 m 和 n 傳給 `run_search`

In [ ]:
settings = {"m/z^0, I^1 (matchms default)": (0, 1), "m/z^0, I^0.5": (0, 0.5),
            "m/z^1, I^0.5 (common)": (1, 0.5), "m/z^3, I^0.6 (Stein&Scott best, EI)": (3, 0.6)}
res_w, rows = {}, []
for name, (mp, ip) in settings.items():
    res_w[name] = run_search(setting=(____, ip))
    ev = evaluate(res_w[name])
    n = ev["n_in_lib"]; p = ev["top1"]
    rows.append({"setting": name, "top1": p, "se": np.sqrt(p * (1 - p) / n), "top3": ev["top3"],
                 "not-in-lib top score >= 0.7": ev["not in lib: top score >= 0.7"]})
wt = pd.DataFrame(rows).set_index("setting")
display(wt.round(3))

<details><summary>💡 練習 5 提示（先自己試，卡住再打開）</summary>

迴圈每一輪的 `(mp, ip)` 就是這一組的 m/z 次方與強度次方；`run_search` 需要一個 `(mz_power, intensity_power)` 的 tuple。

<details><summary>參考答案</summary>

```python
res_w[name] = run_search(setting=(mp, ip))
```
</details>
</details>

**看結果**：四種設定的 top-1 差多少？差距比標準誤（`se`）大嗎？

**但先想一個問題**：前驅物 m/z 篩選已經把候選縮得很小，很多查詢的視窗裡**只有一個化合物**，不管怎麼加權都會選到它，這些查詢拉平了四種設定的差距。真正能看出加權作用的，是**有競爭者**（候選裡至少有 2 個不同化合物）的查詢。下面只看這些。

In [ ]:
# 5-2 只看「庫內、而且候選裡有 ≥ 2 個不同化合物」的查詢
base = res_w["m/z^1, I^0.5 (common)"]
comp_idx = [i for i, r in enumerate(base) if r["in_ref"] and len({t[2] for t in r["top"]}) >= 2]
print(f"庫內查詢 {sum(r['in_ref'] for r in base)} 筆，其中有競爭者的 {len(comp_idx)} 筆")
rows = []
for name, r_list in res_w.items():
    ok = [bool(r_list[i]["top"]) and r_list[i]["top"][0][2] == r_list[i]["true"] for i in comp_idx]
    p = np.mean(ok)
    rows.append({"setting": name, "top1 (competitive only)": p, "se": np.sqrt(p * (1 - p) / len(comp_idx))})
display(pd.DataFrame(rows).set_index("setting").round(3))

注意：只看有競爭者的查詢，**放大的是百分比的差距，不是證據的強度**（同樣的查詢、同樣的對錯，只是分母變小了）；標準誤也跟著變大。

**想一想**
1. 只看有競爭者的查詢，四種設定的差距變大了嗎？「強度開根號」（`I^0.5`）和不開根號（`I^1`）比，差距有沒有大於標準誤？m/z 的次方（0、1、3）呢？
2. 把 `QUERY_LAB` 換成 `"BAFG"` 或 `"LCSB"` 再跑一次（從 Part 1 依序重跑；課堂上可以分組，每組跑一個實驗室再比較）。最好的設定還是同一個嗎？
3. Stein & Scott 的資料是 EI、低解析度，而且**沒有先用前驅物 m/z 篩候選**；這裡是 LC-MS/MS、高解析度，而且篩過。這些差異會怎麼影響結論？

## Part 6. 看失敗案例

**為什麼**：統計數字背後是一個個具體的失敗。看過這些案例，才知道看到高分時要多問什麼。

**怎麼做**：把兩類失敗收集起來（**兩類都套用了「分數 ≥ 0.7」的門檻**，所以只列高分的失敗，不是全部第 1 名錯的）：
1. **化合物在庫裡，但第 1 名是錯的**；
2. **化合物不在庫裡，卻有高分命中**（用規則 `min_matches=1、cutoff=0.7`，也就是只要求分數 ≥ 0.7，不要求匹配峰數）。

然後依對上的峰數與分子式分類。**兩個原因可以同時成立**：例如只對上 1 個峰、而且兩個化合物是同分異構物。

### 練習 6：把失敗案例分類

規則：對上的峰數 **≤ 2** 加上標籤「few peaks」（只靠少數幾個峰）；**分子式相同**加上標籤「isomer」（同分異構物）；兩個都沒有叫「other」。

In [ ]:
fails = []
for r in res:
    t = call(r, 1, 0.7)
    if t is None:
        continue
    score, matches, ik, j = t
    if r["in_ref"] and ik == r["true"]:
        continue                                  # 判斷正確，不是失敗
    kind = "wrong top-1 (in lib)" if r["in_ref"] else "hit but NOT in lib"
    same_formula = query.formula[r["qi"]] == ref.formula[j]
    tags = []
    if matches <= ____:
        tags.append("few peaks")
    if same_formula:
        tags.append("isomer")
    reason = " + ".join(tags) if tags else "other"
    fails.append({"kind": kind, "query": query["name"][r["qi"]], "hit": ref["name"][j], "formula_q": query.formula[r["qi"]],
                  "formula_hit": ref.formula[j], "score": score, "matches": matches, "reason": reason, "qi": r["qi"], "j": j})
fails = pd.DataFrame(fails)
if len(fails):
    fails = fails.sort_values("score", ascending=False).reset_index(drop=True)
print(f"失敗案例 {len(fails)} 筆")
if len(fails):
    display(fails.groupby(["kind", "reason"]).size().rename("n").reset_index())
    display(fails.drop(columns=["qi", "j"]).head(10).round(3))
else:
    print("這個查詢實驗室沒有高分的失敗案例（可以換一個 QUERY_LAB 再看）")

<details><summary>💡 練習 6 提示（先自己試，卡住再打開）</summary>

對上的峰數 `matches` 小於或等於 2 就是 few peaks，所以條件是 `matches <= 2`。

<details><summary>參考答案</summary>

```python
if matches <= 2:
        tags.append("few peaks")
```
</details>
</details>

In [ ]:
# 6-2 畫分數最高的 3 個不同的失敗案例（上：查詢；下：譜庫命中；只畫前處理後的峰）
def plot_mirror(q, l, tol=0.025, title=""):
    fig, ax = plt.subplots(figsize=(7, 3.6))
    lm, li, qm = l.peaks.mz, l.peaks.intensities, q.peaks.mz
    for m, i in zip(qm, q.peaks.intensities):
        ax.vlines(m, 0, i, color="#EE6C4D" if np.any(np.abs(lm - m) <= tol) else "#8D99AE", lw=1.5)
    for m, i in zip(lm, li):
        ax.vlines(m, 0, -i, color="#EE6C4D" if np.any(np.abs(qm - m) <= tol) else "#8D99AE", lw=1.5)
    ax.axhline(0, color="k", lw=0.6); ax.set_ylim(-1.1, 1.1); ax.set_xlabel("m/z"); ax.set_ylabel("relative intensity")
    ax.plot([], [], color="#EE6C4D", lw=2, label="matched"); ax.plot([], [], color="#8D99AE", lw=2, label="not matched")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.set_title(title, fontsize=9); plt.tight_layout(); plt.show()

if len(fails):
    for _, f in fails.drop_duplicates(["query", "hit"]).head(3).iterrows():
        plot_mirror(q_specs[int(f.qi)], ref_specs[int(f.j)],
                    title=f"[{f.reason}] query: {f['query']}  vs  hit: {f.hit}   score {f.score:.3f}, matched peaks {int(f.matches)}")

**想一想**
1. 「few peaks」的失敗，你會用哪一個門檻擋掉？注意圖上**唯一對上的那個峰**：它離前驅物近嗎？（前處理已經把最靠近前驅物的峰移除了。）
2. 「isomer」的失敗，只靠 MS/MS 分數能不能分開？還需要什麼資訊（回想講課的 RT 與標準品）？
3. 有些「錯」其實是**識別碼**的問題（兩個名字相同但 `ik14` 不同，被算成不同化合物）。你怎麼檢查？光靠 InChIKey 前 14 碼夠不夠？
4. 選一個失敗的鏡像圖，用一句話寫下：你會怎麼避免這種失敗？（門檻？RT？標準品？）
5. 假設你在論文裡要報告一個命中，除了分數，你至少要寫哪些資訊？（匹配峰數、tolerance、譜庫來源、信心等級。）

---
# 延伸（課後／有餘力）

## E1. Modified cosine：找「類似物」

**為什麼**：cosine 只比對**m/z 完全相同**的碎片。如果查詢分子和庫裡的分子只差一個小官能基（例如多了一個氧），前驅物質量差 15.995，碎片也跟著位移，普通 cosine 對不上。**Modified cosine** 讓碎片除了原位置，也可以平移「前驅物質量差」來配對，因此適合找**結構類似**的化合物。

In [ ]:
from matchms.similarity import ModifiedCosine
carb = get_spec(lib[lib.name.str.lower() == "carbamazepine"].iloc[0])
hcarb = get_spec(lib[lib.name.str.lower() == "2-hydroxycarbamazepine"].iloc[0])
cos_plain = CosineGreedy(tolerance=MS2_TOL, remove_precursor=False).pair(carb, hcarb)
cos_mod = ModifiedCosine(tolerance=MS2_TOL, remove_precursor=False).pair(carb, hcarb)
print(f"carbamazepine vs 2-hydroxycarbamazepine（多一個氧，前驅物差 {hcarb.metadata['precursor_mz'] - carb.metadata['precursor_mz']:.3f}）")
print(f"  cosine          = {float(cos_plain['score']):.3f}（對上 {int(cos_plain['matches'])} 個峰）")
print(f"  modified cosine = {float(cos_mod['score']):.3f}（對上 {int(cos_mod['matches'])} 個峰）")

**想一想**：modified cosine 分數較高，代表它們是同一個化合物嗎？它比較適合回答什麼問題？

## E2. 熵相似度（spectral entropy similarity）

**為什麼**：Li 等人（2021, Nat Methods）提出用譜的**熵**來比較兩張譜，在大規模測試中優於 dot product 等 42 種演算法。概念：把每張譜的強度正規化成機率（總和 1），把兩張譜合併後算 Shannon 熵 H＝−Σ p ln p，相似度 ＝ 1 − (2·H(合併) − H(A) − H(B)) ÷ ln 4。（完整定義請看原論文。）matchms 內建 `Entropy`。這裡拿它重算 Part 3 的 top-1，和 cosine 比較。**這一格要跑一陣子**（要對每個候選算一次）。

In [ ]:
from matchms.similarity import Entropy

def run_search_entropy(topn=30):
    s = Entropy(tolerance=MS2_TOL, remove_precursor=False)
    out = []
    for qi in range(len(query)):
        q = q_specs[qi]; pmz = query.precursor_mz[qi]
        lo, hi = np.searchsorted(ref_mz, [pmz - MS1_TOL, pmz + MS1_TOL])
        hits = []
        for j in range(lo, hi):
            score = float(s.pair(ref_specs[j], q))
            if score > 0:                                  # 分數 0（完全沒有共同碎片）不算候選
                hits.append((score, 0, ref.ik14[j], j))
        hits.sort(reverse=True)
        out.append({"qi": qi, "true": query.ik14[qi], "in_ref": bool(query.in_ref[qi]), "top": hits[:topn]})
    return out

res_e = run_search_entropy()
inr = [r for r in res_e if r["in_ref"]]
print(f"熵相似度 top-1 = {np.mean([bool(r['top']) and r['top'][0][2] == r['true'] for r in inr]):.3f}"
      f"（cosine 常用設定 = {evaluate(res)['top1']:.3f}）")

**想一想**：兩者的 top-1 差多少？記得 Part 5 的教訓：這個差距有沒有大於抽樣誤差？換成其他查詢實驗室，結論會不會一樣？

## E3. Reverse dot product 與 presence ratio（MS-DIAL 的算法）

**為什麼**：MS-DIAL 用三個分數描述一次比對：

| 分數 | 怎麼算 | 回答什麼 |
|---|---|---|
| dot product | (Σ W查詢·W庫)² ÷ (ΣW查詢² · ΣW庫²)，所有峰都算 | 整張譜像不像 |
| reverse dot product | 同上，但**只用譜庫裡有的峰**（查詢多出來的峰不扣分） | 庫裡的碎片在查詢裡都找得到嗎（查詢可以有雜訊或共流出物的峰） |
| presence ratio | 對上的碎片數 ÷ 譜庫的碎片數 | 庫裡的碎片有幾成找得到 |

下面用最簡單的權重（W＝強度本身，所以 dot product 是 cosine 的**平方**，和 Part 2 的 cosine 不同）實作，並對照一個正確命中和各種失敗案例。

In [ ]:
def ms_dial_like(q, l, tol=MS2_TOL):
    """回傳 (dot product, reverse dot product, presence ratio)；權重 W = 強度。"""
    qm, qi = q.peaks.mz, q.peaks.intensities
    lm, li = l.peaks.mz, l.peaks.intensities
    matched_q = np.zeros(len(lm))                       # 每個譜庫峰在查詢裡對上的強度（沒有就 0）
    for k, m in enumerate(lm):
        idx = np.where(np.abs(qm - m) <= tol)[0]
        if len(idx):
            matched_q[k] = qi[idx[np.argmin(np.abs(qm[idx] - m))]]
    num = np.sum(matched_q * li) ** 2
    dot = num / (np.sum(qi ** 2) * np.sum(li ** 2)) if len(qi) and len(li) else 0.0
    rev = num / (np.sum(matched_q ** 2) * np.sum(li ** 2)) if matched_q.any() else 0.0
    presence = float(np.mean(matched_q > 0)) if len(lm) else 0.0
    return dot, rev, presence

pairs = []
good = next((r for r in res if r["in_ref"] and r["top"] and r["top"][0][2] == r["true"] and r["top"][0][0] >= 0.8), None)
if good is not None:
    pairs.append(("correct hit", q_specs[good["qi"]], ref_specs[good["top"][0][3]], query["name"][good["qi"]], ref["name"][good["top"][0][3]]))
for reason in ("few peaks", "few peaks + isomer", "isomer", "other"):      # 每一種失敗原因各取分數最高的一個
    sub = fails[fails.reason == reason] if len(fails) else fails
    if len(sub):
        f = sub.iloc[0]
        pairs.append((f"failure: {f.reason}", q_specs[int(f.qi)], ref_specs[int(f.j)], f["query"], f["hit"]))
rows = []
for label, q, l, qname, lname in pairs:
    d, rv, pr = ms_dial_like(q, l)
    rows.append({"case": label, "query": qname, "hit": lname, "dot": d, "reverse dot": rv, "presence ratio": pr, "library peaks": len(l.peaks.mz)})
display(pd.DataFrame(rows).round(3))

**想一想**
1. 失敗案例的三個分數，和正確命中比起來是比較高還是比較低？（注意看 `library peaks`：譜庫那張譜有幾個峰？）
2. 為什麼 reverse dot product 和 presence ratio，對「譜庫那張譜只有 1–2 個峰」的情況特別寬鬆？要擋住這種假命中，靠的是哪一個條件？（提示：Part 4 的 `min_matches`。）
3. 把三個分數都放進判斷規則，你會怎麼設計？

---
**下一步**：回到講課，把你的結果對照信心等級：只靠譜庫比對、沒有標準品，最多能報告到第幾級？